# Python Data Analysis & ML Foundations

This notebook is a small portfolio project where I practice the core Python tools used in data analysis and machine learning.

The project covers:

- NumPy for numerical computing and matrix operations
- pandas for tabular data processing
- matplotlib and seaborn for data visualization
- scipy for mathematical optimization
- scikit-learn for model training, preprocessing, and validation

The goal is not only to solve exercises, but also to show a clear workflow: from basic numerical operations to a simple machine learning model.


## 0. Environment Setup

I start by importing the main libraries that will be used throughout the notebook. I also set a random seed to make the results reproducible.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize, minimize_scalar

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")


## 1. Understanding Machine Learning Basics

Before working with code, I briefly summarize the main ideas behind machine learning and the libraries used in this project.


### 1.1 What types of machine learning tasks exist?

From a mathematical point of view, the most common machine learning tasks are:

1. **Classification**  
   The target variable is categorical. The model predicts a class label.  
   Example: predict whether a passenger survived or not.

2. **Regression**  
   The target variable is numerical. The model predicts a continuous value.  
   Example: predict house price, salary, temperature, or sales.

3. **Clustering**  
   There is no target variable. The model groups similar objects together.  
   Example: customer segmentation.

4. **Dimensionality reduction**  
   The model reduces the number of features while trying to keep the most important information.  
   Example: PCA for visualization or feature compression.

5. **Anomaly detection**  
   The model searches for unusual objects that are different from normal data.  
   Example: fraud detection or unusual network activity.

6. **Reinforcement learning**  
   An agent learns to make decisions by interacting with an environment and receiving rewards or penalties.


### 1.2 What are the main Python ML libraries used for?

- **NumPy** — fast numerical computations, arrays, matrices, linear algebra.
- **pandas** — working with tables, CSV files, filtering, grouping, cleaning data.
- **matplotlib** — basic plotting and chart customization.
- **seaborn** — statistical visualization built on top of matplotlib.
- **scipy** — scientific computing, optimization, statistics, numerical methods.
- **scikit-learn** — machine learning models, preprocessing, pipelines, metrics, validation.
- **streamlit** — building simple interactive web apps and ML demos.


### 1.3 Why do we split data into train and test sets?

The train/test split helps us understand how well a model works on new unseen data.

- **Train set** is used to fit the model.
- **Test set** is used to estimate model quality on data that the model has not seen before.

If we evaluate only on training data, the result can be too optimistic because the model may simply memorize the training examples. This problem is called **overfitting**.


### 1.4 kNN question: can classification change when k changes from 2 to 3?

For standard unweighted kNN with two classes, if an object is classified as class **A** unambiguously with `k=2`, it means both of the two nearest neighbors belong to class **A**.

If we increase `k` to `3`, the third neighbor can belong to class **B**, but the vote will still be:

- A: 2 votes
- B: 1 vote

So the predicted class will remain **A**.

However, if we use weighted voting, custom tie-breaking rules, or if the `k=2` result was not truly unambiguous, the prediction may change.


### 1.5 What to do when code returns an error?

My debugging plan:

1. Read the full error message carefully.
2. Find the exact line where the error happened.
3. Check variable names, types, shapes, and missing values.
4. Print intermediate results with `print()`, `.head()`, `.shape`, `.info()`.
5. Simplify the code and test it step by step.
6. Read the documentation for the function that failed.
7. Search for the error message if the problem is still unclear.
8. Fix one issue at a time and rerun the cell.


## 2. NumPy: Matrix and Vector Operations

In this section, I practice basic NumPy operations that are important for machine learning:

- creating arrays and matrices
- calculating statistics
- transposing matrices
- element-wise operations
- matrix multiplication
- sorting and indexing vectors


### 2.1 Creating a random matrix


In [ ]:
# Create a random 10x10 matrix with integer values from 0 to 100 inclusive
A = rng.integers(0, 101, size=(10, 10))
A


**Conclusion:**  
The matrix `A` is a two-dimensional NumPy array. In machine learning, matrices are often used to store datasets where rows are observations and columns are features.


### 2.2 Column means


In [ ]:
# Calculate mean values for each column
column_means = A.mean(axis=0)
column_means


**Conclusion:**  
`axis=0` means that NumPy calculates the statistic down the rows, so we get one mean value for each column.


### 2.3 Matrix transpose


In [ ]:
# Transpose matrix A
A_T = A.T
A_T


**Conclusion:**  
Transposing swaps rows and columns. This operation is often used in linear algebra and machine learning algorithms.


### 2.4 Element-wise multiplication


In [ ]:
# Element-wise multiplication of A and its transpose
elementwise_product = A * A_T
elementwise_product


**Conclusion:**  
Element-wise multiplication multiplies values at the same positions. It is different from matrix multiplication.


### 2.5 Matrix multiplication


In [ ]:
# Matrix multiplication: A multiplied by A
B = A @ A

# Alternative syntax:
# B = np.dot(A, A)

B


In [ ]:
# Check that matrix multiplication is equivalent to matrix power A^2
assert np.array_equal(B, np.linalg.matrix_power(A, 2))
print("Matrix multiplication check passed.")


**Conclusion:**  
The `@` operator performs matrix multiplication. This operation is one of the foundations of machine learning, especially in linear models and neural networks.


### 2.6 Creating and shuffling a vector


In [ ]:
# Create a vector with 20 consecutive integer values from 1 to 20
v = np.arange(1, 21)
v


In [ ]:
# Shuffle the vector in place
rng.shuffle(v)
v


In [ ]:
assert v is not None
print("Vector was created and shuffled.")


**Conclusion:**  
`shuffle` changes the array in place. This means it modifies the original array and does not return a new one.


### 2.7 Maximum value and its index


In [ ]:
# Maximum value in the vector
max_value = v.max()
max_value


In [ ]:
# Index of the maximum value
max_ind = v.argmax()
max_ind


In [ ]:
assert v[max_ind] == v.max()
print("Maximum index check passed.")


**Conclusion:**  
`argmax()` returns the index of the maximum element. This is useful when we need not only the best value, but also its position.


### 2.8 Sorting vector in descending order


In [ ]:
# Sort vector in descending order
v_sorted = np.sort(v)[::-1]
v_sorted


In [ ]:
assert np.array_equal(v_sorted, np.arange(20, 0, -1))
print("Sorting check passed.")


**Conclusion:**  
Sorting helps organize numerical data. In this case, I sorted the vector from the largest value to the smallest.


## 3. Data Visualization and Mathematical Optimization

In this section, I visualize mathematical functions and use SciPy to find minimum points.


### 3.1 Plotting one function with matplotlib


In [ ]:
def f(x):
    return 3 * x**6 - 5 * x**4 + 2 * x - 1

x = np.linspace(-2, 2, 500)
y_f = f(x)

plt.figure(figsize=(9, 5))
plt.plot(x, y_f, label=r"$f(x)=3x^6-5x^4+2x-1$")
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Function plot")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True)
plt.show()


**Conclusion:**  
The graph helps understand the shape of the function and visually estimate where local minimum points may be located.


### 3.2 Plotting two functions on one chart


In [ ]:
def g(x):
    return -x**5 + 5 * x**3 - 2 * x**2 + 7

y_g = g(x)

plt.figure(figsize=(9, 5))
plt.plot(x, y_f, label=r"$f(x)=3x^6-5x^4+2x-1$")
plt.plot(x, y_g, label=r"$g(x)=-x^5+5x^3-2x^2+7$")
plt.axhline(0, linewidth=1)
plt.axvline(0, linewidth=1)
plt.title("Two functions on one plot")
plt.xlabel("x")
plt.ylabel("value")
plt.legend()
plt.grid(True)
plt.show()


**Conclusion:**  
Plotting multiple functions on the same chart makes it easier to compare their behavior over the same interval.


### 3.3 Finding local minima with SciPy


In [ ]:
# Find one minimum for each function on a fixed interval
f_min = minimize_scalar(f, bounds=(-2, 2), method="bounded")
g_min = minimize_scalar(g, bounds=(-2, 2), method="bounded")

minima_df = pd.DataFrame({
    "function": ["f(x)", "g(x)"],
    "x_min": [f_min.x, g_min.x],
    "function_value": [f_min.fun, g_min.fun],
    "success": [f_min.success, g_min.success]
})

minima_df


**Conclusion:**  
SciPy can numerically search for minimum points. Here I used `minimize_scalar` for one-dimensional functions on the interval `[-2, 2]`.


### 3.4 Optimization of a function with two variables


In [ ]:
def h(values):
    x, y = values
    return 3 * x * y + 3 * x**2 - 4 * y + 7 - 5 * x**5

# Important note:
# Without bounds this function has no global minimum,
# because it can go to negative infinity.
# For a practical numerical example, I minimize it on a bounded domain.

bounds = [(-2, 2), (-2, 2)]
starting_points = [
    (0, 0),
    (1, 1),
    (-1, 1),
    (2, -2),
]

results = []
for point in starting_points:
    result = minimize(h, x0=np.array(point), bounds=bounds, method="L-BFGS-B")
    results.append(result)

best_result = min(results, key=lambda result: result.fun)

print("Best point:", best_result.x)
print("Minimum value on bounded domain:", best_result.fun)
print("Optimization success:", best_result.success)


**Conclusion:**  
For this two-variable function, there is no global unconstrained minimum. To make the optimization practical, I searched for the minimum inside the bounded region `x ∈ [-2, 2]` and `y ∈ [-2, 2]`.


## 4. Statistical Visualization with Seaborn

In this section, I use the `tips` dataset and visualize a 60% confidence interval for tips depending on the day of the week.


In [ ]:
def load_tips_dataset():
    """Load seaborn tips dataset. If internet/cache is unavailable, create a small fallback dataset."""
    try:
        data = sns.load_dataset("tips")
        source = "seaborn tips dataset"
    except Exception:
        # Fallback only keeps the notebook executable when seaborn datasets are not available.
        local_rng = np.random.default_rng(RANDOM_STATE)
        days = np.repeat(["Thur", "Fri", "Sat", "Sun"], [60, 20, 80, 70])
        base_tip = {"Thur": 2.8, "Fri": 2.7, "Sat": 3.0, "Sun": 3.2}
        tips_values = [max(0.5, local_rng.normal(base_tip[day], 0.8)) for day in days]
        total_bill = [max(5, local_rng.normal(18 + base_tip[day] * 2, 6)) for day in days]
        data = pd.DataFrame({
            "day": days,
            "tip": tips_values,
            "total_bill": total_bill,
        })
        source = "synthetic fallback tips-like dataset"
    return data, source


tips, tips_source = load_tips_dataset()
print("Dataset source:", tips_source)
tips.head()


In [ ]:
plt.figure(figsize=(8, 5))

# seaborn versions differ: new versions use errorbar=, older versions use ci=
try:
    sns.barplot(data=tips, x="day", y="tip", errorbar=("ci", 60))
except TypeError:
    sns.barplot(data=tips, x="day", y="tip", ci=60)

plt.title("60% Confidence Interval for Tip by Day")
plt.xlabel("Day")
plt.ylabel("Tip")
plt.show()


**Conclusion:**  
The chart compares average tip values across weekdays and shows uncertainty using a 60% confidence interval. Confidence intervals are useful because they show not only the mean value but also how stable the estimate is.


## 5. Titanic Classification with kNN

In this section, I build a simple machine learning pipeline for the Titanic survival prediction task.

The workflow is:

1. Load the dataset.
2. Clean and prepare features.
3. Build preprocessing steps for numerical and categorical columns.
4. Train a kNN classifier.
5. Evaluate the model with cross-validation.
6. Optionally create a Kaggle submission file if `test.csv` is available.

Raw Kaggle CSV files are usually not committed to GitHub. They should be placed locally in the `data/` folder.


### 5.1 Loading Titanic data


In [ ]:
def load_titanic_data():
    """Load Titanic data from local files or from a public CSV mirror."""
    data_dir = Path("../data")
    train_path = data_dir / "train.csv"
    test_path = data_dir / "test.csv"

    if train_path.exists():
        train = pd.read_csv(train_path)
        test = pd.read_csv(test_path) if test_path.exists() else None
        source = "local Kaggle files from ../data/"
    else:
        # Public mirror of the Titanic training dataset.
        # If you use Kaggle directly, place train.csv and test.csv into ../data/ instead.
        url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
        train = pd.read_csv(url)
        test = None
        source = "public Titanic CSV mirror"

    return train, test, source


train_df, test_df, titanic_source = load_titanic_data()
print("Dataset source:", titanic_source)
print("Train shape:", train_df.shape)
if test_df is not None:
    print("Test shape:", test_df.shape)

train_df.head()


**Conclusion:**  
The dataset contains passenger information such as class, sex, age, fare, family size, and survival status. The target variable is `Survived`.


### 5.2 Basic dataset inspection


In [ ]:
train_df.info()


In [ ]:
train_df.isna().sum().sort_values(ascending=False)


**Conclusion:**  
The dataset contains missing values, especially in columns such as `Age`, `Cabin`, and sometimes `Embarked`. These values must be handled before training a model.


### 5.3 Feature engineering


In [ ]:
def add_titanic_features(df):
    """Create additional features for the Titanic dataset."""
    df = df.copy()

    # Family-related features
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    # Extract title from passenger name
    if "Name" in df.columns:
        df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
        df["Title"] = df["Title"].fillna("Unknown")
        rare_titles = [
            "Lady", "Countess", "Capt", "Col", "Don", "Dr", "Major", "Rev", "Sir", 
            "Jonkheer", "Dona"
        ]
        df["Title"] = df["Title"].replace(rare_titles, "Rare")
        df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    else:
        df["Title"] = "Unknown"

    # Extract deck from cabin
    if "Cabin" in df.columns:
        df["Deck"] = df["Cabin"].astype(str).str[0]
        df.loc[df["Cabin"].isna(), "Deck"] = "Unknown"
    else:
        df["Deck"] = "Unknown"

    return df


train_features_df = add_titanic_features(train_df)
train_features_df[["Survived", "Pclass", "Sex", "Age", "Fare", "FamilySize", "IsAlone", "Title", "Deck"]].head()


**Conclusion:**  
I created new features such as `FamilySize`, `IsAlone`, `Title`, and `Deck`. Feature engineering can improve model quality because it gives the algorithm more useful information.


### 5.4 Preparing X and y


In [ ]:
TARGET = "Survived"

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]
categorical_features = ["Sex", "Embarked", "Title", "Deck"]

X = train_features_df[numeric_features + categorical_features]
y = train_features_df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
X.head()


**Conclusion:**  
The features are separated into numerical and categorical columns. This is important because they require different preprocessing steps.


### 5.5 Building a preprocessing and kNN pipeline


In [ ]:
# Compatibility for different scikit-learn versions
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", one_hot_encoder)
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("knn", KNeighborsClassifier())
])

knn_pipeline


**Conclusion:**  
The pipeline makes the workflow clean and reproducible. It combines missing value handling, scaling, one-hot encoding, and model training in one object.


### 5.6 Cross-validation baseline


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

baseline_scores = cross_val_score(knn_pipeline, X, y, cv=cv, scoring="accuracy")

print("Baseline CV scores:", np.round(baseline_scores, 3))
print("Baseline mean accuracy:", baseline_scores.mean().round(3))


**Conclusion:**  
Cross-validation gives a more reliable estimate than a single train/test split because the model is trained and evaluated several times on different parts of the data.


### 5.7 Hyperparameter tuning


In [ ]:
param_grid = {
    "knn__n_neighbors": list(range(3, 32, 2)),
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],
}

grid_search = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
)

grid_search.fit(X, y)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", round(grid_search.best_score_, 3))


**Conclusion:**  
Tuning `n_neighbors`, distance weighting, and distance metric helps improve the kNN classifier. The goal is to achieve accuracy above `0.7`, which is realistic for this dataset with basic preprocessing and feature engineering.


### 5.8 Creating a Kaggle submission file


In [ ]:
if test_df is not None:
    test_features_df = add_titanic_features(test_df)
    X_test = test_features_df[numeric_features + categorical_features]

    best_model = grid_search.best_estimator_
    test_predictions = best_model.predict(X_test)

    submission = pd.DataFrame({
        "PassengerId": test_df["PassengerId"],
        "Survived": test_predictions
    })

    output_path = Path("../submission_knn_titanic.csv")
    submission.to_csv(output_path, index=False)

    print(f"Submission file saved to: {output_path}")
    display(submission.head())
else:
    print("Kaggle test.csv was not found. Place test.csv into ../data/ to create a submission file.")


**Conclusion:**  
If the Kaggle `test.csv` file is available, the notebook creates a valid submission file with `PassengerId` and `Survived` columns.


## 6. Final Project Summary

In this project, I practiced the core tools used in Python data analysis and introductory machine learning.

What I practiced:

- NumPy arrays, matrix operations, vector operations, sorting, indexing
- pandas data inspection and feature preparation
- matplotlib function visualization
- seaborn statistical visualization with confidence intervals
- scipy numerical optimization
- scikit-learn preprocessing, pipelines, cross-validation, and kNN classification

Main learning outcome:

> A machine learning project is not only about training a model. It also includes data loading, inspection, preprocessing, feature engineering, validation, and clear communication of results.

This notebook can be used as an introductory portfolio project that demonstrates practical familiarity with the Python ML ecosystem.
